In [1]:
from IPython.display import Image

$$
\text{理论带宽}=\frac{\text{数据传输速率}\times\text{总线位宽}}{8}
$$


### Apple M chip

- SoC 与 UMA
    - 在传统的 x86 架构（如大多数搭载 Intel 或 AMD 处理器的个人电脑）中，计算机的核心计算组件是高度物理分离的。主板（Motherboard）扮演着庞大城市路网的角色：
        - CPU（中央处理器）、独立 GPU（图形处理器） 和 RAM（内存，system ram） 作为独立的物理组件，插在主板的不同插槽上。
        - 组件之间必须通过主板上的物理铜线轨迹和总线（如 PCIe 总线）进行数据通信。
    - Apple M 系列芯片所采用的 SoC 架构，是一种极致的集成化设计。
        - 它将 CPU、GPU、统一内存（Unified Memory）、神经网络引擎（NPU）、安全隔区（Secure Enclave）以及各种输入输出控制器等原本分散的组件，全部光刻在同一块微小的硅片（Die）上，或封装在极其紧凑的同一个封装系统（SiP）内。
        - UMA：
            - 传统架构的性能损耗，很大一部分来自于数据搬运。例如在剪辑视频时，CPU 读取了系统内存（RAM）中的视频数据，若要交给独立显卡渲染，必须将庞大的数据通过 PCIe 总线复制到显存（VRAM）中。这不仅耗时，而且极度耗电。
            - M 芯片引入了 UMA (Unified Memory Architecture)。由于 CPU 和 GPU 共享同一块高速内存，数据只需存入一次。当 GPU 需要处理 CPU 刚刚解压的数据时，它只需读取同一个内存地址即可——这被称为**“零拷贝（Zero-copy）”**。这是 M 芯片在视频剪辑和 3D 渲染等重度任务中表现出惊人效率的核心秘密。
- nvidia gpu vs. apple m 芯片
    - Nvidia 的高端 GPU 是一张巨大的专属硅片，上面 90% 以上的面积都密密麻麻地铺满了用于纯粹数学计算的 ALU（算术逻辑单元）和 Tensor Core，以及匹配的高速缓存。它生来就是一台纯粹的“并行计算机器”。& vram (gddr, hbm)
    - M 芯片是一个系统级芯片（SoC）。在这块有限的硅片上，GPU 核心只能分到一部分领地。Apple 必须将剩余的大量面积划拨给强大的中央处理器（CPU）、专门的神经网络引擎（Neural Engine）、媒体处理引擎（用于视频编解码）、图像信号处理器（ISP）以及极其庞大的统一内存控制器。
- fusion architecture 
    - M5 Pro and M5 Max are built using the new Apple-designed **Fusion Architecture** and engineered from the ground up for AI. This innovative design combines two dies into a single system on a chip, providing tremendous performance boosts.
        - M5 Pro 和 M5 Max 不再是一整块硅片，而是由“两颗较小的晶粒（Die）+ 高速通信总线”拼接而成的融合体。
    - 把两块硅片（two dies）用先进封装与高速互连“封装级融合”成一颗逻辑上的 SoC，让软件把它当作“一颗芯片”来用。(Chiplet)
        - Die（硅片/裸片）：一块实际刻出来的硅。过去很多 SoC 主要是“单一大 die”。
    - 对比 M3 ultrafusion
        - M1/M3 ultra 用 UltraFusion 把两颗 M1 Max 连接起来（Symmetrical Joining，对称拼接），借助硅中介层（silicon interposer）跨 1 万+ 信号提供 2.5TB/s 级别的互连带宽，让软件把它当作“一个芯片”。
            - https://www.apple.com/at/newsroom/2022/03/apple-unveils-m1-ultra-the-worlds-most-powerful-chip-for-a-personal-computer/
        - m5 fusion architecture
            - 芯片内部的“解构与重组”，(Asymmetrical Dies)

In [15]:
from IPython.display import Image

In [18]:
# https://x.com/Frederic_Orange/status/2029160827190677695/photo/1
Image(url='./imgs/m5-pro-max-Architect.jpeg', width=700)

### lpddr

- lpddr5
    - 6.4 (6400mt/s)
- https://my.avnet.com/silica/products/new-products/npi/2024/micron-lpddr5x/
    - 9.6 Gbps

### GPU core (tensor core)

- cuda cores vs. gpu cores
    - 5090 有 21,760 个 CUDA 核心，而 8 or 10 cores on the base M5, 16 or 20 on the M5 Pro, and 32 or 40 on the M5 Max.
    - gpu core 它的物理体量和逻辑层级，实际上对应的是 Nvidia 的一个完整 “SM”（或 AMD 的 Compute Unit）。在 Apple 的 1 个 GPU Core 内部，同样包罗了大量的执行单元（ALUs）、纹理单元和独立缓存。
        - Nvidia 的“CUDA Core”： 实际上是 GPU 内部最微小的算术逻辑单元（ALU），它负责执行最基础的单点浮点运算。Nvidia 将成百上千个这样的微小 ALU 打包成一个“流多处理器”（SM - Streaming Multiprocessor）。以 RTX 5090 为例，其内部实际上是 170 个 SM。 (21760 / 170 = 128)
- The M5 generation introduces a next-generation GPU architecture with a dedicated Neural Accelerator integrated into each GPU core.
    - M5 features a faster CPU and next-generation GPU with a Neural Accelerator in each core, enabling MacBook Air to power through a variety of workflows, from creative projects to complex AI tasks.
    - neural accelerator
        - 也就是在 M5 之前，GPU 主要负责图形渲染和通用的并行计算。
        - 它提供 专用的矩阵乘法（matrix-multiplication）运算，而矩阵乘法正是很多机器学习负载的核心。
        - https://machinelearning.apple.com/research/exploring-llms-mlx-m5
- gpu core vs. npu (neural engine)
    - 独立的神经处理单元（NPU），拥有自己的多核架构（如 16 核）。追求极致的能效比（Performance-per-Watt）。专门处理操作系统的常驻 AI 任务和 Apple Intelligence 的日常后台功能。例如：实时的语音识别、Face ID 验证、输入法的预测性文本、照片的后台特征索引、视频通话中的背景虚化等。即使全天候运行这些任务，Neural Engine 也只会消耗极低的电量，不会对设备的电池续航造成显著负担。

### m5 bandwidth

- 三种芯片方案
    - 15/18 core cpu + 16 core GPU, bitwidth = 128 + 128 = 256
        - 9.6 * 256 / 8 = 307
    - 18 core cpu + 32 core GPU, bitwidth = 128 + (128 + 128) = 384
        - 9.6 * 384 / 8 = 460
    - 18 core cpu + 40 core GPU, bitwidth = (128 + 128) + (128 + 128) = 512
        - 9.6 * 512 / 8 = 614

### m3

- M3 Ultra Mac Studio
    - 内存带宽： > 800 GB/s。
    - 起步容量： 64GB 或 128GB (视具体配置而定)
        - 最大可选容量： 512 GB
- https://www.apple.com/newsroom/2023/10/apple-unveils-m3-m3-pro-and-m3-max-the-most-advanced-chips-for-a-personal-computer/

In [2]:
Image(url='https://www.apple.com/newsroom/images/2023/10/Apple-unveils-M3-M3-Pro-and-M3-Max/article/Apple-M3-chip-series-architecture-231030_big.jpg.large_2x.jpg', 
      width=400)

In [13]:
# m3
# 2个内存颗粒
Image(url='https://www.apple.com/newsroom/images/2023/10/Apple-unveils-M3-M3-Pro-and-M3-Max/article/Apple-M3-chip-series-unified-memory-architecture-M3-231030_big.jpg.large.jpg', 
      width=600)

In [12]:
# m3 pro
# 3个内存颗粒
Image(url='https://www.apple.com/newsroom/images/2023/10/Apple-unveils-M3-M3-Pro-and-M3-Max/article/Apple-M3-chip-series-unified-memory-architecture-M3-Pro-231030_big.jpg.large.jpg', 
      width=600)

In [14]:
# m3 max
# 4个内存颗粒
Image(url='https://www.apple.com/newsroom/images/2023/10/Apple-unveils-M3-M3-Pro-and-M3-Max/article/Apple-M3-chip-series-unified-memory-architecture-M3-Max-231030_big.jpg.large.jpg', 
      width=600)

### misc

- dram 普遍涨价的时代，为什么苹果内存条还能保持相对平稳的价格
    - 涨价的真凶是“HBM 挤出效应”，而非消费端需求爆发
        - 当前 DRAM 涨价的根本推手，是 AI 数据中心对 HBM（高带宽内存） 和服务器级 DDR5 内存的狂热需求。三星、美光和 SK 海力（Samsung, Micron, SK Hynix）士为了赚取 HBM 的超高利润，将大量原本用于生产消费级、普通 DDR4 的晶圆产能（Wafer Starts）强行转移到了 HBM 上。
    - 锁价机制： 苹果并不在现货市场上按月采购内存。作为全球最大的消费电子巨头之一，苹果通过庞大的订单量与三星（Samsung）、SK海力士（SK Hynix）和美光（Micron）等寡头供应商签订长期协议（Long-Term Agreements）。这些协议通常在几个月甚至一两年之前就已经提前锁定了价格和产能。
    - 缓冲效应：苹果长期以来在内存升级上收取着远超行业平均水平的溢价（常被业界戏称为“苹果税”）。例如，在 Mac 产线中，将内存从 8GB 升级到 16GB 通常需要消费者额外支付 200 美元（或 1500 元人民币），而该容量 LPDDR 内存的实际物料成本（BOM）可能仅为十几美元。
    - 自 Apple Silicon（M系列芯片）全面推行以来，苹果采用了统一内存架构（Unified Memory Architecture）。LPDDR 内存芯片不再是主板上可插拔的独立模块，而是直接与 SoC（系统级芯片）紧密封装在同一个基板上。
        - 从“大宗商品”到“定制组件”： 这一技术转变使得苹果采购的不再是标准化的、受市场波动影响极大的内存条，而是深度定制的 LPDDR 颗粒及先进封装服务。高度的定制化进一步削弱了标准 DRAM 现货价格波动对苹果硬件的直接冲击，因为内存已经成为了芯片整体制造成本的一部分。